# 📊 Retail Sales Data Cleaning & Feature Engineering Notebook

This Jupyter notebook demonstrates end-to-end data cleaning, missing value imputation, duplicate removal, data type casting, filtering, and feature engineering using **Pandas** and **NumPy**.

In [1]:
%pip install -q pandas numpy

import pandas as pd
import numpy as np

# Set display options for better visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

Note: you may need to restart the kernel to use updated packages.


## Step 1: Read the Raw CSV File
Read `sales_data.csv` specifying `header=2` because the first 2 rows contain metadata headers.

In [2]:
df_raw = pd.read_csv('sales_data.csv', header=2)
print('Initial Dataset Shape:', df_raw.shape)
df_raw.head()

Initial Dataset Shape: (27, 12)


,order_id,customer_name,category,product,city,quantity,unit_price,order_date,status,sales,profit,discount
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1001.0,Aarav,Electronics,Laptop,Kathmandu,2.0,85000.0,1/5/2026,Completed,170000.0,34000.0,17000.0
2,1002.0,Sita,Furniture,Office Chair,Lalitpur,1.0,18000.0,1/8/2026,Completed,18000.0,3600.0,900.0
3,1003.0,Rohan,Electronics,Headphones,Bhaktapur,3.0,4500.0,1/12/2026,Pending,13500.0,2700.0,675.0
4,1004.0,Anisha,Clothing,Jacket,Kathmandu,2.0,6500.0,1/15/2026,Completed,13000.0,2600.0,650.0


## Step 2: Clean and Rename Columns
Standardize column names: strip leading/trailing whitespace, convert to lowercase, and replace spaces with underscores.

In [3]:
df = df_raw.copy()
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print('Cleaned Column Names:', df.columns.tolist())
df.head()

Cleaned Column Names: ['order_id', 'customer_name', 'category', 'product', 'city', 'quantity', 'unit_price', 'order_date', 'status', 'sales', 'profit', 'discount']


,order_id,customer_name,category,product,city,quantity,unit_price,order_date,status,sales,profit,discount
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1001.0,Aarav,Electronics,Laptop,Kathmandu,2.0,85000.0,1/5/2026,Completed,170000.0,34000.0,17000.0
2,1002.0,Sita,Furniture,Office Chair,Lalitpur,1.0,18000.0,1/8/2026,Completed,18000.0,3600.0,900.0
3,1003.0,Rohan,Electronics,Headphones,Bhaktapur,3.0,4500.0,1/12/2026,Pending,13500.0,2700.0,675.0
4,1004.0,Anisha,Clothing,Jacket,Kathmandu,2.0,6500.0,1/15/2026,Completed,13000.0,2600.0,650.0


## Step 3: Drop Completely Empty Rows & Cast Numeric Data Types
Remove rows where all values are NaN, and ensure numeric columns are properly converted to numerical data types.

In [4]:
rows_all_null = df.isnull().all(axis=1)
print(f'Completely empty rows found: {rows_all_null.sum()}')

df = df.dropna(how='all').reset_index(drop=True)

# Convert numeric columns
numeric_cols = ['order_id', 'quantity', 'unit_price', 'sales', 'profit', 'discount']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df.head()

Completely empty rows found: 1


,order_id,customer_name,category,product,city,quantity,unit_price,order_date,status,sales,profit,discount
0,1001.0,Aarav,Electronics,Laptop,Kathmandu,2.0,85000.0,1/5/2026,Completed,170000.0,34000.0,17000.0
1,1002.0,Sita,Furniture,Office Chair,Lalitpur,1.0,18000.0,1/8/2026,Completed,18000.0,3600.0,900.0
2,1003.0,Rohan,Electronics,Headphones,Bhaktapur,3.0,4500.0,1/12/2026,Pending,13500.0,2700.0,675.0
3,1004.0,Anisha,Clothing,Jacket,Kathmandu,2.0,6500.0,1/15/2026,Completed,13000.0,2600.0,650.0
4,1005.0,Bibek,Electronics,Mobile Phone,Pokhara,1.0,55000.0,1/20/2026,Cancelled,55000.0,11000.0,5500.0


## Step 4: Drop Unnecessary Columns
**Justification:** Drop column `'city'` as it contains missing values and is non-essential for financial order & revenue analysis.

In [5]:
df = df.drop(columns=['city'])
df.head()

,order_id,customer_name,category,product,quantity,unit_price,order_date,status,sales,profit,discount
0,1001.0,Aarav,Electronics,Laptop,2.0,85000.0,1/5/2026,Completed,170000.0,34000.0,17000.0
1,1002.0,Sita,Furniture,Office Chair,1.0,18000.0,1/8/2026,Completed,18000.0,3600.0,900.0
2,1003.0,Rohan,Electronics,Headphones,3.0,4500.0,1/12/2026,Pending,13500.0,2700.0,675.0
3,1004.0,Anisha,Clothing,Jacket,2.0,6500.0,1/15/2026,Completed,13000.0,2600.0,650.0
4,1005.0,Bibek,Electronics,Mobile Phone,1.0,55000.0,1/20/2026,Cancelled,55000.0,11000.0,5500.0


## Step 5: Check Missing Values Count
Count missing values across all remaining columns.

In [6]:
missing_counts = df.isnull().sum()
print('Missing values per column:')
print(missing_counts)

Missing values per column:
order_id         0
customer_name    1
category         0
product          0
quantity         1
unit_price       1
order_date       0
status           1
sales            3
profit           3
discount         3
dtype: int64


## Step 6: Impute & Handle Missing Numeric Values
1. Drop rows where critical value `'sales'` is missing.
2. Fill missing `'profit'` with the **mean** profit.
3. Fill missing `'discount'` with the **median** discount.

In [7]:
# Drop missing sales
sales_missing_count = df['sales'].isnull().sum()
df = df.dropna(subset=['sales']).reset_index(drop=True)
print(f'Dropped {sales_missing_count} row(s) missing sales.')

# Impute profit and discount
profit_mean = df['profit'].mean()
discount_median = df['discount'].median()

print(f'Mean profit calculated: {profit_mean:.2f}')
print(f'Median discount calculated: {discount_median:.2f}')

df['profit'] = df['profit'].fillna(profit_mean)
df['discount'] = df['discount'].fillna(discount_median)

print('\nMissing values remaining in numeric columns:')
print(df[['sales', 'profit', 'discount']].isnull().sum())

Dropped 3 row(s) missing sales.
Mean profit calculated: 7323.64
Median discount calculated: 825.00

Missing values remaining in numeric columns:
sales       0
profit      0
discount    0
dtype: int64


## Step 7: Handle Missing Categorical Data
Fill missing `'customer_name'` values with `'Unknown'`.

In [8]:
missing_cust_before = df['customer_name'].isnull().sum()
df['customer_name'] = df['customer_name'].fillna('Unknown')
print(f'Replaced {missing_cust_before} missing customer_name value(s) with Unknown.')

Replaced 1 missing customer_name value(s) with Unknown.


## Step 8: Detect & Remove Duplicates
Identify duplicate orders based on `order_id` and keep only the first occurrence.

In [9]:
duplicates_mask = df.duplicated(subset=['order_id'], keep=False)
print('Duplicate rows identified:')
display(df[duplicates_mask][['order_id', 'customer_name', 'category', 'product', 'sales']])

num_duplicates = df.duplicated(subset=['order_id'], keep='first').sum()
df = df.drop_duplicates(subset=['order_id'], keep='first').reset_index(drop=True)
print(f'\nRemoved {num_duplicates} duplicate row(s) based on order_id.')

Duplicate rows identified:


,order_id,customer_name,category,product,sales
0,1001.0,Aarav,Electronics,Laptop,170000.0
6,1007.0,Nabin,Furniture,Desk,50000.0
11,1013.0,Roshan,Electronics,Monitor,56000.0
18,1001.0,Aarav,Electronics,Laptop,170000.0
19,1007.0,Nabin,Furniture,Desk,50000.0
20,1013.0,Roshan,Electronics,Monitor,56000.0



Removed 3 duplicate row(s) based on order_id.


## Step 9: Data Filtering & Feature Engineering
1. **Filter 1:** Orders with `unit_price > 20,000`.
2. **Filter 2:** Completed orders with `unit_price > 10,000`.
3. **Feature 1 (`total_amount`):** `quantity * unit_price - discount`
4. **Feature 2 (`customer_type`):** `'Bulk Buyer'` if `quantity >= 3` else `'Regular Buyer'`

In [10]:
print('--- Filter 1: Orders with unit_price > 20,000 ---')
high_unit_price = df[df['unit_price'] > 20000]
display(high_unit_price[['order_id', 'customer_name', 'product', 'unit_price']])

print('\n--- Filter 2: Completed orders with unit_price > 10,000 ---')
filter2 = df[(df['unit_price'] > 10000) & (df['status'] == 'Completed')]
display(filter2[['customer_name', 'category', 'status', 'unit_price']])

# Create new feature columns
df['total_amount'] = df['quantity'] * df['unit_price'] - df['discount']
df['customer_type'] = np.where(df['quantity'] >= 3, 'Bulk Buyer', 'Regular Buyer')

print('\nSample rows with new feature columns:')
display(df[['order_id', 'customer_name', 'quantity', 'unit_price', 'discount', 'total_amount', 'customer_type']].head())

--- Filter 1: Orders with unit_price > 20,000 ---


,order_id,customer_name,product,unit_price
0,1001.0,Aarav,Laptop,85000.0
4,1005.0,Bibek,Mobile Phone,55000.0
6,1007.0,Nabin,Desk,25000.0
11,1013.0,Roshan,Monitor,28000.0
15,1017.0,Manoj,Table,22000.0



--- Filter 2: Completed orders with unit_price > 10,000 ---


,customer_name,category,status,unit_price
0,Aarav,Electronics,Completed,85000.0
1,Sita,Furniture,Completed,18000.0
6,Nabin,Furniture,Completed,25000.0
11,Roshan,Electronics,Completed,28000.0
15,Manoj,Furniture,Completed,22000.0
17,Rita,Electronics,Completed,12000.0
18,Bikash,Furniture,Completed,18000.0



Sample rows with new feature columns:


,order_id,customer_name,quantity,unit_price,discount,total_amount,customer_type
0,1001.0,Aarav,2.0,85000.0,17000.0,153000.0,Regular Buyer
1,1002.0,Sita,1.0,18000.0,900.0,17100.0,Regular Buyer
2,1003.0,Rohan,3.0,4500.0,675.0,12825.0,Bulk Buyer
3,1004.0,Anisha,2.0,6500.0,650.0,12350.0,Regular Buyer
4,1005.0,Bibek,1.0,55000.0,5500.0,49500.0,Regular Buyer


## Step 10: Final Clean Dataset Verification & Export
Compare dataset shape before and after cleaning, inspect final output, and export to CSV.

In [11]:
print(f'Dataset Shape BEFORE cleaning: {df_raw.shape}')
print(f'Dataset Shape AFTER cleaning:  {df.shape}')

print('\nFirst 10 rows of final cleaned dataset:')
display(df.head(10))

# Export clean dataset to CSV
df.to_csv('cleaned_sales_data.csv', index=False)
print('\nCleaned dataset saved to cleaned_sales_data.csv.')

Dataset Shape BEFORE cleaning: (27, 12)
Dataset Shape AFTER cleaning:  (20, 13)

First 10 rows of final cleaned dataset:


,order_id,customer_name,category,product,quantity,unit_price,order_date,status,sales,profit,discount,total_amount,customer_type
0,1001.0,Aarav,Electronics,Laptop,2.0,85000.0,1/5/2026,Completed,170000.0,34000.0,17000.0,153000.0,Regular Buyer
1,1002.0,Sita,Furniture,Office Chair,1.0,18000.0,1/8/2026,Completed,18000.0,3600.0,900.0,17100.0,Regular Buyer
2,1003.0,Rohan,Electronics,Headphones,3.0,4500.0,1/12/2026,Pending,13500.0,2700.0,675.0,12825.0,Bulk Buyer
3,1004.0,Anisha,Clothing,Jacket,2.0,6500.0,1/15/2026,Completed,13000.0,2600.0,650.0,12350.0,Regular Buyer
4,1005.0,Bibek,Electronics,Mobile Phone,1.0,55000.0,1/20/2026,Cancelled,55000.0,11000.0,5500.0,49500.0,Regular Buyer
5,1006.0,Pragya,Books,Python Book,4.0,2200.0,1/22/2026,Completed,8800.0,1760.0,440.0,8360.0,Bulk Buyer
6,1007.0,Nabin,Furniture,Desk,2.0,25000.0,1/25/2026,Completed,50000.0,10000.0,5000.0,45000.0,Regular Buyer
7,1008.0,Samir,Clothing,Sneakers,3.0,7500.0,2/2/2026,Pending,22500.0,4500.0,1125.0,21375.0,Bulk Buyer
8,1010.0,Kiran,Books,Data Science Book,2.0,3500.0,2/9/2026,Completed,7000.0,1400.0,350.0,6650.0,Regular Buyer
9,1011.0,Mina,Clothing,T-Shirt,5.0,1800.0,2/12/2026,Completed,9000.0,1800.0,450.0,8550.0,Bulk Buyer



Cleaned dataset saved to cleaned_sales_data.csv.
